# Week 48, Day 2: Motion illusions as optimal percepts

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w048/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Slower at low contrast

With true speed 1, prior sd 1 and likelihood sd 0.1 / contrast, compute the posterior mean speed for contrasts 0.05, 0.1, 0.2, 0.5 and 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
contrasts = np.array([0.05, 0.1, 0.2, 0.5, 1.0])
sigma_p, sigma_0, v_true = 1.0, 0.1, 1.0
sigma_l = sigma_0 / contrasts
v_hat = ...
for c, v in zip(contrasts, v_hat):
    print(f"contrast {c:4.2f}: perceived speed {v:.2f}")
plt.semilogx(contrasts, v_hat, "o-"); plt.xlabel("contrast"); plt.ylabel("perceived speed / true speed"); plt.show()

In [ ]:
import numpy as np
def test_monotonic():
    "perceived speed increases with contrast"
    assert all(np.diff(v_hat) > 0)
def test_ratio():
    "at the lowest contrast perceived speed is below half the true speed, at the highest above 90%"
    assert v_hat[0] < 0.5 and v_hat[-1] > 0.9

test_monotonic()
test_ratio()
print("All checks passed")

### Exercise: Perceived direction against contrast

Write map_v(sigma): build N from the two normals, s = N v_true, and solve the linear system with prior sd 1. Compute the perceived direction (in degrees) for sigma 0.02, 0.1, 0.3, 1 and 3, and the vector average direction of the two normal velocities.

In [ ]:
v_true = np.array([1.0, 0.0])
angles = np.deg2rad([50, 75])
N = np.stack([np.cos(angles), np.sin(angles)], 1)
s = N @ v_true

def map_v(sigma, sigma_p=1.0):
    A = ...
    b = ...
    return np.linalg.solve(A, b)

sigmas = [0.02, 0.1, 0.3, 1.0, 3.0]
vs = [map_v(sg) for sg in sigmas]
dirs = np.array([np.degrees(np.arctan2(v[1], v[0])) for v in vs])
va = (s[:, None] * N).mean(0); va_dir = np.degrees(np.arctan2(va[1], va[0]))
for sg, v, d in zip(sigmas, vs, dirs):
    print(f"noise {sg:4.2f}: perceived velocity {np.round(v, 3)}, direction {d:5.1f} deg, speed {np.linalg.norm(v):.2f}")
print(f"vector average direction {va_dir:.1f} deg; intersection of constraints 0 deg")

In [ ]:
import numpy as np
def test_ioc():
    "at the smallest noise the percept is within 2 degrees of the true direction (IOC)"
    assert abs(dirs[0]) < 2
def test_va():
    "at the largest noise it is within 5 degrees of the vector average direction"
    assert abs(dirs[-1] - va_dir) < 5
def test_monotone():
    "perceived direction rotates steadily as noise grows"
    assert all(np.diff(dirs) > 0)

test_ioc()
test_va()
test_monotone()
print("All checks passed")